In [11]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    r"C:\JOB_market_trend_analysis\raw_data\job_postings_fact.csv"
)

print("=" * 60)
print("Dataset Loaded Successfully")
print("Shape:", df.shape)
print("=" * 60)

print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Dataset Loaded Successfully
Shape: (787686, 16)

Columns:
['job_id', 'company_id', 'job_title_short', 'job_title', 'job_location', 'job_via', 'job_schedule_type', 'job_work_from_home', 'search_location', 'job_posted_date', 'job_no_degree_mention', 'job_health_insurance', 'job_country', 'salary_rate', 'salary_year_avg', 'salary_hour_avg']


,job_id,company_id,job_title_short,job_title,job_location,job_via,job_schedule_type,job_work_from_home,search_location,job_posted_date,job_no_degree_mention,job_health_insurance,job_country,salary_rate,salary_year_avg,salary_hour_avg
0,0,0,Data Analyst,Marketing Data Analyst,Anywhere,via LinkedIn,Full-time,True,Serbia,25-09-2023 17:46,False,False,Serbia,NaN,NaN,NaN
1,55,1,Cloud Engineer,Storage and Virtualization Engineer,"Kuwait City, Kuwait",via Trabajo.org,Full-time,False,Kuwait,30-07-2023 17:49,True,False,Kuwait,NaN,NaN,NaN
2,66,2,Data Analyst,Data Analyst et Scientist F/H,"Paris, France",via Emplois Trabajo.org,Full-time,False,France,28-07-2023 17:28,False,False,France,NaN,NaN,NaN
3,76,3,Data Engineer,Data Engineer,"Denver, CO",via LinkedIn,Contractor,False,"Illinois, United States",03-04-2023 17:14,False,False,United States,hour,NaN,70.0
4,81,4,Data Engineer,Data Engineer,Anywhere,via LinkedIn,Contractor,True,Canada,25-03-2023 17:25,False,False,Canada,NaN,NaN,NaN


In [9]:
print("Before:", df.shape)




Before: (787686, 16)


In [17]:
import pandas as pd

df = pd.read_csv(r"C:\JOB_market_trend_analysis\raw_data\job_postings_fact_.csv")

# Step 1: total postings for each calendar month (Jan–Dec), across all years combined
# — this captures seasonality (e.g. "January is always a hiring surge") rather than
# one-off year-month spikes.
month_counts = df.groupby('posted_month').size().rename('postings_count_by_month')
df['postings_count_by_month'] = df['posted_month'].map(month_counts)

# Step 2: rank the 12 months by volume — 1 = highest-hiring month
month_rank = month_counts.rank(ascending=False, method='min').astype(int)
df['month_hiring_rank'] = df['posted_month'].map(month_rank)

# Step 3: a simple, defensible "high demand" flag — above the 12-month average
overall_avg = month_counts.mean()
df['is_high_demand_month'] = df['postings_count_by_month'] > overall_avg

# Step 4 (optional but useful for a dashboard slicer): 3-tier label instead of
# a flat True/False, based on tertiles of the 12 monthly totals
tertile_bounds = month_counts.quantile([1/3, 2/3])
def tier(count):
    if count >= tertile_bounds.iloc[1]:
        return 'High'
    elif count >= tertile_bounds.iloc[0]:
        return 'Medium'
    else:
        return 'Low'
month_tier = month_counts.apply(tier)
df['hiring_demand_tier'] = df['posted_month'].map(month_tier)

print(df[['posted_month', 'postings_count_by_month', 'month_hiring_rank', 'is_high_demand_month', 'hiring_demand_tier']].drop_duplicates().sort_values('posted_month'))

df.to_csv('job_postings_cleaned_v4.csv', index=False)

    posted_month  postings_count_by_month  month_hiring_rank  \
5              1                    92266                  1   
7              2                    64560                  4   
4              3                    64158                  6   
3              4                    62915                  8   
16             5                    52235                 12   
10             6                    61500                 10   
1              7                    63855                  7   
6              8                    75067                  2   
0              9                    62433                  9   
8             10                    66601                  3   
21            11                    64404                  5   
20            12                    57692                 11   

    is_high_demand_month hiring_demand_tier  
5                   True               High  
7                  False               High  
4                  False     

In [20]:
df = pd.read_csv(r"C:\JOB_market_trend_analysis\cleaned_data\job_postings_cleaned_v4.csv")
df.head(20)


,job_id,company_id,job_title_short,job_title,job_location,job_via,job_schedule_type,job_work_from_home,search_location,job_posted_date,...,seniority_level,work_mode,posted_year,posted_month,posted_quarter,skill_count_per_posting,postings_count_by_month,month_hiring_rank,is_high_demand_month,hiring_demand_tier
0,0,0,Data Analyst,Marketing Data Analyst,Anywhere,via LinkedIn,Full-time,True,Serbia,25-09-2023 17:46,...,Mid/Unspecified,Remote,2023,9,3,5,62433,9,False,Low
1,55,1,Cloud Engineer,Storage and Virtualization Engineer,"Kuwait City, Kuwait",via Trabajo.org,Full-time,False,Kuwait,30-07-2023 17:49,...,Mid/Unspecified,Hybrid/In-Office,2023,7,3,1,63855,7,False,Medium
2,66,2,Data Analyst,Data Analyst et Scientist F/H,"Paris, France",via Emplois Trabajo.org,Full-time,False,France,28-07-2023 17:28,...,Mid/Unspecified,Hybrid/In-Office,2023,7,3,8,63855,7,False,Medium
3,76,3,Data Engineer,Data Engineer,"Denver, CO",via LinkedIn,Contractor,False,"Illinois, United States",03-04-2023 17:14,...,Mid/Unspecified,Hybrid/In-Office,2023,4,2,5,62915,8,False,Medium
4,81,4,Data Engineer,Data Engineer,Anywhere,via LinkedIn,Contractor,True,Canada,25-03-2023 17:25,...,Mid/Unspecified,Remote,2023,3,1,3,64158,6,False,Medium
5,105,5,Data Analyst,Data Analyst,"Bangkok, Thailand",via Jobtopgun.com,Full-time,False,Thailand,29-01-2023 17:16,...,Mid/Unspecified,Hybrid/In-Office,2023,1,1,0,92266,1,True,High
6,106,6,Data Engineer,Data Lead Engineer (with strong Python) - Remo...,Anywhere,via Jobgether,Full-time,True,Nicaragua,13-08-2023 17:37,...,Lead,Remote,2023,8,3,6,75067,2,True,High
7,116,7,Senior Data Engineer,Senior Data Engineer,"New York, NY",via Melga,Full-time,False,"Florida, United States",19-02-2023 17:11,...,Senior,Hybrid/In-Office,2023,2,1,2,64560,4,False,High
8,122,8,Data Analyst,Full Time Data Analyst,"Chicago, IL",via Trabajo.org,Full-time,False,"Illinois, United States",19-10-2023 17:01,...,Mid/Unspecified,Hybrid/In-Office,2023,10,4,1,66601,3,True,High
9,134,9,Data Engineer,Data Engineer Remote / Telecommute Jobs,Anywhere,via Clearance Jobs,Full-time,True,Georgia,27-04-2023 17:38,...,Mid/Unspecified,Remote,2023,4,2,2,62915,8,False,Medium


In [19]:
df.describe()

,job_id,company_id,salary_year_avg,salary_hour_avg,salary_standardized,posted_year,posted_month,posted_quarter,skill_count_per_posting,postings_count_by_month,month_hiring_rank
count,7.876860e+05,787686.000000,22034.000000,10665.000000,32699.000000,787686.000000,787686.000000,787686.000000,787686.000000,787686.000000,787686.000000
mean,7.514609e+05,97968.549456,123268.815643,47.045199,114979.606036,2022.997594,6.311996,2.454197,4.658714,67017.352161,6.088778
std,5.302434e+05,160374.990244,48271.063013,21.933562,48896.432623,0.048990,3.531373,1.134953,4.041610,10470.918463,3.492483
min,0.000000e+00,0.000000,15000.000000,8.000000,15000.000000,2022.000000,1.000000,1.000000,0.000000,52235.000000,1.000000
25%,2.785158e+05,3332.000000,90000.000000,27.500000,81682.000000,2023.000000,3.000000,1.000000,2.000000,62433.000000,3.000000
50%,6.727035e+05,23326.000000,115000.000000,46.000000,111175.000000,2023.000000,6.000000,2.000000,4.000000,64158.000000,6.000000
75%,1.185031e+06,112107.000000,150000.000000,61.159996,145500.000000,2023.000000,9.000000,3.000000,7.000000,66601.000000,9.000000
max,1.826678e+06,787681.000000,960000.000000,391.000000,960000.000000,2023.000000,12.000000,4.000000,53.000000,92266.000000,12.000000
